In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load cleaned dataset
df = pd.read_csv("cleaned_sales.csv")

# Convert Date column
df["Date"] = pd.to_datetime(df["Date"])

print("Dataset Loaded Successfully")
print(df.head())

Dataset Loaded Successfully
     Sales_Person    Country              Product       Date  Amount  \
0  Jehu Rudeforth         UK      Mint Chip Choco 2022-01-04    5320   
1     Van Tuxwell      India        85% Dark Bars 2022-08-01    7896   
2    Gigi Bohling      India  Peanut Butter Cubes 2022-07-07    4501   
3    Jan Morforth  Australia  Peanut Butter Cubes 2022-04-27   12726   
4  Jehu Rudeforth         UK  Peanut Butter Cubes 2022-02-24   13685   

   Boxes_Shipped  
0            180  
1             94  
2             91  
3            342  
4            184  


In [3]:
# Year
df["Year"] = df["Date"].dt.year

# Month
df["Month"] = df["Date"].dt.month

# Day
df["Day"] = df["Date"].dt.day

# Day Name
df["Day_Name"] = df["Date"].dt.day_name()

# Day of Week
df["Day_of_Week"] = df["Date"].dt.dayofweek

# Week Number
df["Week"] = df["Date"].dt.isocalendar().week

# Quarter
df["Quarter"] = df["Date"].dt.quarter

# Weekend Flag
df["Weekend"] = df["Day_of_Week"].apply(lambda x: 1 if x >= 5 else 0)

In [4]:
df["Revenue_per_Box"] = df["Amount"] / df["Boxes_Shipped"]

In [5]:
product_encoder = LabelEncoder()
country_encoder = LabelEncoder()
sales_encoder = LabelEncoder()

df["Product_ID"] = product_encoder.fit_transform(df["Product"])

df["Country_ID"] = country_encoder.fit_transform(df["Country"])

df["Salesperson_ID"] = sales_encoder.fit_transform(df["Sales_Person"])

In [6]:
df = df.sort_values(["Product", "Date"])

df.reset_index(drop=True, inplace=True)

In [7]:
# Previous day's demand
df["Lag_1"] = (
    df.groupby("Product")["Boxes_Shipped"]
      .shift(1)
)

# Previous week's demand
df["Lag_7"] = (
    df.groupby("Product")["Boxes_Shipped"]
      .shift(7)
)

In [8]:
df["Rolling_7"] = (
    df.groupby("Product")["Boxes_Shipped"]
      .transform(lambda x: x.rolling(7).mean())
)

df["Rolling_30"] = (
    df.groupby("Product")["Boxes_Shipped"]
      .transform(lambda x: x.rolling(30).mean())
)

In [9]:
# Fill missing Values
df.fillna(0, inplace=True)

In [10]:
print("\nNew Columns")

print(df.columns)

print("\nShape")

print(df.shape)

print(df.head())


New Columns
Index(['Sales_Person', 'Country', 'Product', 'Date', 'Amount', 'Boxes_Shipped',
       'Year', 'Month', 'Day', 'Day_Name', 'Day_of_Week', 'Week', 'Quarter',
       'Weekend', 'Revenue_per_Box', 'Product_ID', 'Country_ID',
       'Salesperson_ID', 'Lag_1', 'Lag_7', 'Rolling_7', 'Rolling_30'],
      dtype='object')

Shape
(1094, 22)
      Sales_Person    Country         Product       Date  Amount  \
0  Gunar Cockshoot     Canada  50% Dark Bites 2022-01-04    3024   
1     Jan Morforth         UK  50% Dark Bites 2022-01-12    5250   
2      Van Tuxwell        USA  50% Dark Bites 2022-01-13    9737   
3  Gunar Cockshoot         UK  50% Dark Bites 2022-01-13    2107   
4   Jehu Rudeforth  Australia  50% Dark Bites 2022-01-14    5194   

   Boxes_Shipped  Year  Month  Day   Day_Name  ...  Quarter  Weekend  \
0             23  2022      1    4    Tuesday  ...        1        0   
1            293  2022      1   12  Wednesday  ...        1        0   
2            160  2022      1

In [12]:
df.to_csv("featured_sales.csv", index=False)

print("\nFeature Engineering Completed Successfully")

print("File Saved : featured_sales.csv")


Feature Engineering Completed Successfully
File Saved : featured_sales.csv


In [14]:
import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Load dataset
df = pd.read_csv("featured_sales.csv")

features = [
    "Amount",
    "Product_ID",
    "Country_ID",
    "Salesperson_ID",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Quarter",
    "Weekend",
    "Revenue_per_Box",
    "Lag_1",
    "Lag_7",
    "Rolling_7",
    "Rolling_30"
]

X = df[features]
y = df["Boxes_Shipped"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

best_model = None
best_r2 = float("-inf")

for name, model in models.items():

    model.fit(X_train, y_train)

    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = mean_squared_error(y_test, prediction) ** 0.5
    r2 = r2_score(y_test, prediction)

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE : {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²  : {r2:.4f}")

    if r2 > best_r2:
        best_r2 = r2
        best_model = model

os.makedirs("models", exist_ok=True)

joblib.dump(
    best_model,
    "models/demand_model.pkl"
)

print("\nBest Model Saved Successfully!")


Linear Regression
----------------------------------------
MAE : 82.94
RMSE: 111.60
R²  : 0.0998

Decision Tree
----------------------------------------
MAE : 24.80
RMSE: 52.43
R²  : 0.8013

Random Forest
----------------------------------------
MAE : 15.11
RMSE: 36.79
R²  : 0.9022

Best Model Saved Successfully!


In [17]:
 predict.py

NameError: name 'predict' is not defined